Limitation of Linear Regression

Can not use for classification.

Why?
There’s a hidden assumption in linear regression is that we assume that the data points are roughly aligned. If the points are arranged on a curve, or scattered around randomly, then we cannot approximate them with a line. And without the line, we cannot make a prediction. The same reasoning applies with multiple input variables: whether we draw a line, a plane, or a higher-dimensional space, we need points that can be reasonably approximated by that shape. Otherwise, we cannot use linear regression.



![My Image](Images/Classification%20graph.png)

Even by looking at this one input variable, we can guess that linear regression would not work well on categorical data such as this one. How are we supposed to draw that line if the points are not aligned at all?

Not only is linear regression a bad approximation for categorical data, it’s also an unstable approximation. To understand what that means, imagine that Roberto’s data contain an outlier, a data point that’s very far away from the others. Maybe there was one day when the pizzeria had a very high number of reservations, but the police were busy with a major operation in the city and never arrived on the scene. If we add this outlier example to the dataset, the line generated by linear regression moves around a lot, as shown below:

![My Image](Images/Impact%20of%20outlier.png)

As we can see, the presence of the outlier leads linear regression to generate a very different line that results in very different predictions. So not only is this model a bad approximation of the points,it’s also extremely sensitive to anomalous data.

Long story short, linear regression is not a good fit for categorical data such as this one.

Invasion of the Sigmoids

To solve above problem, we need some wrapper function that squashes values between 0 to 1 and handle outliers. That's where sigmoid helps:

^y = wrapper_function(x1*w1 + x2*w2 + x3*w3 + ...)

what the wrapper_function() does? It takes any number out of the weighted sum, and squashes it into the range from 0 to 1.

The other requirement is that the function that we look for should work well with gradient descent. Think about the following:

We use this function to calculate ^y.
Then we use ^y to calculate the loss.
Finally, we descend the loss with gradient descent.
For gradient descent, the wrapper function should be smooth, without flat areas (where the gradient drops to zero) or gaps (where the gradient is not even defined).

For the sake of gradient descent, the wrapper function should be smooth, without flat areas (where the gradient drops to zero) or gaps (where the gradient is not even defined).

To wrap it up, we want a function that smoothly changes across the range from 0 to 1
without ever jumping or flatlining. Something like this:

![My Image](Images/Sigmoid.png)


sigmoid’s formula:

σ(z) = 1 / (1 + e ** (-z))

the sigmoid squeezes any value to a narrow band ranging from 0 to 1. it doesn’t have any steep cliffs, and it never goes completely flat. That’s the function we need!
 

In [14]:
import numpy as np

In [15]:
def sigmoid(z):
    return 1/(1+ np.exp(-z))


In [16]:
#prediction or forward pass

def forward(X,w):
    weighted_sum = np.matmul(X,w)
    return sigmoid(weighted_sum)

'''
The result of forward() is our prediction ^y, which is a matrix with the same dimensions 
as the weighted sum, that is, one row per example and one column. 
Each element in the matrix is now constrained between 0 and 1 only. 
More closer to 0 or 1 more confident on prediction.
'''

'\nThe result of forward() is our prediction ^y, which is a matrix with the same dimensions \nas the weighted sum, that is, one row per example and one column. \nEach element in the matrix is now constrained between 0 and 1 only. \nMore closer to 0 or 1 more confident on prediction.\n'

In [17]:
'''
During the training phase, that gradual variation in confidence is just what we need. 
We want the loss to change smoothly to slide over it with gradient descent. 
However, we want the system to get straight to the point once we switch from 
the training phase to the classification phase. The labels that we use to train 
the classifier are either 0 or 1, so the classification should also be a straight 
0 or 1. To get that unambiguous answer, during the classification phase, 
we can round the result to the nearest integer, like this:
'''

def classify(X,w):
    return np.round(forward(X,w))

Smooting it out

We can not use sigmoid with MSE loss. It will create problem. 

See those deep canyons leading straight into holes? We mentioned those holes when we introduced gradient descent—they are the dreaded local minima. Remember that the goal of gradient descent is to move downhill? Now consider what happens if GD enters a local minimum: since there is no “downhill” at the bottom of a hole, the algorithm stops there, falsely convinced that it’s reached the global minimum that it was aiming for.

By looking at this diagram, we can conclude that if we use the mean squared error and the sigmoid together, the resulting loss has an uneven surface littered with local minima. Such a surface is hard to navigate with gradient descent. We’d better look for a different loss function with a smoother, more GD-friendly surface.

![My Image](Images/MSE%20loss%20issue%20with%20Sigmoid.png)

Log loss
          m 
L= (−1/m) ∑ (y * log(^y) + (1-y) * log(1-^y))
         i=1

Remember that each label in the matrix Y is either 0 or 1. For labels that are 
0, the first_term is multiplied by 0, so it disappears. For labels that are 
1, the second_term disappears because it’s multiplied by (1−Y). So each element of 
Y contributes only one of the two terms.

Let’s plot the log loss and see what it looks like:

![My Image](Images/log%20loss.png)

In [18]:
'''It is nice and smooth! There are no canyons, flat areas, or holes. 
From now on, this will be our loss function.'''
def loss(X,Y,w):
    y_hat = forward(X,w)
    first_term = Y* np.log(y_hat)
    second_term = (1-Y) * np.log(1-y_hat)
    return -np.average(first_term + second_term)

Update the Gradient

partial derivation of log loss:
              m
δL/δw = (1/m) ∑ x * (^y - y)
             i=1 

In [19]:
def gradient(X,Y,w):
    return np.matmul(X.T, (forward(X,w) - Y)) / X.shape[0]

![My Image](Images/Trained%20sigmoid%20function.png)

Below a certain threshold (which in this example seems to be around 12), the model’s value is under 0.5, and classify() rounds it to 0. Above that threshold, the mode’s value is over 0.5 and classify() rounds it to 1. The result is a stepwise model function. The job of the training phase is to stretch and shift this curve and ultimately set the threshold where the output switches from 0 to 1.

As we add more features in input this 2-D graph turned into more dimensional...
However, concept stays the same.

In [20]:
#Trianing the model
def train(X,Y,iterations,lr):
    w = np.zeros((X.shape[1],1))
    for i in range(iterations):
        if (i%2000==0 or i==9999):
            print("Iteration %4d => Loss: %.20f" % (i,loss(X,Y,w)))
        w -=gradient(X,Y,w)*lr
    return w


In [21]:
#inference phase

def test(X,Y,w):
    total_examples = X.shape[0]
    correct_results = np.sum(classify(X,w) == Y)
    success_percent = correct_results*100 / total_examples

    print("\nSuccess: %d/%d (%.20f)" % (correct_results,total_examples, success_percent))

In [22]:
#Prepare data

x1,x2,x3,y = np.loadtxt("Datasets/police.txt",skiprows=1,unpack=True)
X = np.column_stack((np.ones(x1.size),x1,x2,x3))
Y = y.reshape(-1,1)
w = train(X,Y,10000,0.001)

test(X,Y,w)



Iteration    0 => Loss: 0.69314718055994528623
Iteration 2000 => Loss: 0.06544781617471000235
Iteration 4000 => Loss: 0.03623187570335883317
Iteration 6000 => Loss: 0.02538915575612082920
Iteration 8000 => Loss: 0.01967526759903708011
Iteration 9999 => Loss: 0.01612744378365649212

Success: 11/11 (100.00000000000000000000)


As we move from linear regression to classification, most functions in our program have to change. We have a brand-new sigmoid() function. The old predict() split into two separate functions: forward(), which is used during training, and classify(), which is used for classification.

We also change the formula that we use to calculate the loss and its gradient: instead of the mean squared error, we use the log loss. As a result, we have brand-new implementations for loss() and gradient().

We also write a new test() function that prints the percentage of correct classifications. The instruction np.sum(classify(X, w) == Y) means that first, compare the predicted values to the labels, returning an array that contains True for the elements that match and False for those that do not. Then, count the True elements.

While some code changes since linear regression, the core concepts do not. The loss() function still tells us how wrong we are. The train() function does not change at all, and it still finds the weights by descending the loss gradient. Finally, we still use those weights in classify() (formerly called predict()) to make a classification.

In [23]:
print(w)

[[-0.16642539]
 [ 1.20413327]
 [-0.56547161]
 [ 0.29781193]]


first weight is bias. Rest 3 are for input variable. We can see some weights are larger than others and some weights are even negative. Weights tells importance of feature but I need to learn how?